In [3]:
import requests
import json
import urllib3

# Desactivación de advertencias de certificados SSL para entornos estatales
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


BASE_URL = "https://datosabiertos.pronabec.gob.pe"
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

# FASE 1: VALIDACIÓN DEL ENDPOINT OFICIAL DE LA DOCUMENTACIÓN

print("--- FASE 1: Verificación de la API oficial ---")
api_oficial_url = "https://api.datosabiertos.gob.pe/developer/NotasDeBecarios?apiKey=YOUR_API_KEY"

try:
    print(f"Evaluando conexión con el endpoint público: {api_oficial_url}")
    respuesta_oficial = requests.get(api_oficial_url, headers=HEADERS, timeout=8, verify=False)
    
    print(f"Código de estado HTTP recibido: {respuesta_oficial.status_code}")
    if respuesta_oficial.status_code != 200:
        print("Resultado: Error de acceso confirmado. El servidor no responde o la ruta es inválida.")
    else:
        print("Resultado: El servidor respondió 200, pero el formato no corresponde a datos estructurados.")
        
except Exception as e:
    print(f"Resultado: Error crítico de conexión. Detalle: {e}")
    print("Conclusión: El dominio de la API oficial se encuentra inoperable o no existe en los servidores DNS.")

print("\n" + "="*70 + "\n")

# FASE 2: COMPROBACIÓN DE ENDPOINTS INTERNOS IDENTIFICADOS (jqGrid)

print("--- FASE 2: Verificación de endpoints internos (jqGrid) ---")
# Diccionario con las rutas de los controladores internos del portal
endpoints_internos = {
    "Notas de Becarios": f"{BASE_URL}/Dataset/ListarNotasDeBecarios",
    "Nota Promedio por Región": f"{BASE_URL}/Dataset/ListarNotaPromedioDelPostulantePorRegion",
    "Beca 18 por Provincia": f"{BASE_URL}/Dataset/ListarBeca18BecariosPorProvincia",
    "Becarios por País de Estudio": f"{BASE_URL}/Dataset/ListarBecariosPorPaisDeEstudio"
}

# Parámetros requeridos por el backend para la paginación y estructura jqGrid
# Se limita la consulta a 5 registros para comprobar la validez de la respuesta
parametros_jqgrid = {
    'sidx': '',      
    'sord': 'asc',   
    'page': '1',     
    'rows': '5'      
}

for nombre_dataset, url_destino in endpoints_internos.items():
    print(f"Evaluando endpoint interno: {nombre_dataset}")
    print(f"Ruta: {url_destino}")
    
    try:
        # Ejecución de la consulta simulando los parámetros de la interfaz web
        respuesta_interna = requests.get(url_destino, params=parametros_jqgrid, headers=HEADERS, timeout=10, verify=False)
        print(f"Código de estado HTTP recibido: {respuesta_interna.status_code}")
        
        if respuesta_interna.status_code == 200:
            try:
                # Conversión de la respuesta a formato JSON para validar la estructura
                datos_recibidos = respuesta_interna.json()
                print("Resultado: Conexión exitosa. El endpoint procesa y devuelve datos estructurados.")
                
                # Extracción del volumen total de registros reportados por el backend
                total_registros = datos_recibidos.get('records', 'No especificado')
                print(f"Volumen de datos en el servidor: {total_registros} registros.")
                
            except json.JSONDecodeError:
                print("Resultado: El servidor respondió con éxito pero la respuesta no contiene un JSON estructurado.")
        else:
            print(f"Resultado: El controlador interno denegó el acceso. Código: {respuesta_interna.status_code}")
            
    except Exception as e:
        print(f"Resultado: Error en la petición. Detalle: {e}")
        print("Nota: Un error de tiempo de espera (timeout) indica un volumen de datos masivo sin responder en el límite de tiempo.")
        
    print("-" * 70)

--- FASE 1: Verificación de la API oficial ---
Evaluando conexión con el endpoint público: https://api.datosabiertos.gob.pe/developer/NotasDeBecarios?apiKey=YOUR_API_KEY
Resultado: Error crítico de conexión. Detalle: HTTPSConnectionPool(host='api.datosabiertos.gob.pe', port=443): Max retries exceeded with url: /developer/NotasDeBecarios?apiKey=YOUR_API_KEY (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x000001EDA8542FB0>: Failed to establish a new connection: [Errno 11001] getaddrinfo failed'))
Conclusión: El dominio de la API oficial se encuentra inoperable o no existe en los servidores DNS.


--- FASE 2: Verificación de endpoints internos (jqGrid) ---
Evaluando endpoint interno: Notas de Becarios
Ruta: https://datosabiertos.pronabec.gob.pe/Dataset/ListarNotasDeBecarios
Resultado: Error en la petición. Detalle: HTTPSConnectionPool(host='datosabiertos.pronabec.gob.pe', port=443): Read timed out. (read timeout=10)
Nota: Un error de tiempo de espera (timeou